# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [76]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [77]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [78]:
import os
from glob import glob

# Write your code below.

import os
import glob

# Load the environment variables
PRICE_DATA = os.getenv('SRC_DIR')
#PRICE_DATA = os.getenv('C:/Users/lionl/dsi/production/05_src/data/features')

if PRICE_DATA is None:
    raise ValueError("SRC_DIR environment variable is not set")

# Construct the data directory path
data_dir = os.path.join(PRICE_DATA, "data")

# Use glob to find all parquet files in the data directory
parquet_files = glob.glob(os.path.join(data_dir, "*/*/*.parquet"))

# Print the results
print(f"Found {len(parquet_files)} parquet files in {data_dir}:")
for file_path in parquet_files:
    print(f"  - {file_path}")

Found 150 parquet files in ../../05_src/data:
  - ../../05_src/data\features\stock_features\part.0.parquet
  - ../../05_src/data\features\stock_features\part.1.parquet
  - ../../05_src/data\features\stock_features\part.10.parquet
  - ../../05_src/data\features\stock_features\part.11.parquet
  - ../../05_src/data\features\stock_features\part.12.parquet
  - ../../05_src/data\features\stock_features\part.13.parquet
  - ../../05_src/data\features\stock_features\part.14.parquet
  - ../../05_src/data\features\stock_features\part.15.parquet
  - ../../05_src/data\features\stock_features\part.16.parquet
  - ../../05_src/data\features\stock_features\part.17.parquet
  - ../../05_src/data\features\stock_features\part.18.parquet
  - ../../05_src/data\features\stock_features\part.19.parquet
  - ../../05_src/data\features\stock_features\part.2.parquet
  - ../../05_src/data\features\stock_features\part.20.parquet
  - ../../05_src/data\features\stock_features\part.21.parquet
  - ../../05_src/data\featu

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [79]:
from glob import glob


PRICE_DATA_px = dd.read_parquet(parquet_files).set_index("ticker")

In [80]:
PRICE_DATA_shift = PRICE_DATA_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

C:\Users\lionl\AppData\Local\Temp\ipykernel_30244\3335384907.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  PRICE_DATA_shift = PRICE_DATA_px.groupby('ticker', group_keys=False).apply(


In [81]:
PRICE_DATA_rets = PRICE_DATA_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

In [82]:
PRICE_DATA_rets = PRICE_DATA_shift.assign(
    
                  hi_lo_range=lambda x: x['High'] - x['Low']
                                )

In [83]:
dd_feat=PRICE_DATA_rets

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [86]:
pandas_df = dd_feat.head(len(dd_feat), compute=True)
pandas_df['returns_ma_10'] = pandas_df['returns'].rolling(10).mean()

KeyError: "['Year', 'Close_lag_1'] not in index"

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
# Answer : It's not necessary to convert to pandas to create a rolling mean. Dash can handle it directly. If we are considering to store everything in memory, then pandas is fine. but if we are working with large datasets that don't fit into memory, then Dash is better choice. Dask for large-scale data loading and filtering, pandas for complex transformations and final aggregations on manageable-sized data

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.